In [3]:
# ================================
# FULL PREPROCESSING CELL (Py3.10)
# ================================

import pandas as pd
import numpy as np
import warnings
import joblib
import warnings
import os

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")

# --- 1) Load data ---
DATA_PATH = "clean_student_data_v2.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print("Shape:", df.shape)

# --- 2) Define target & drop columns ---
TARGET_COLUMN = "raw_score"

# Giữ đúng tinh thần bạn đang drop:
# - year, student_id: định danh/metadata
# - current_semester_gpa, cumulative_gpa: có thể gây leakage tùy cách bạn tạo
# - previous_*: tùy bạn, nhưng bạn đang muốn drop để model tập trung vào FE mới
# - expected_difficulty, subject_type, fail_rate_*: bạn đang loại bỏ ở pipeline hiện tại
DROP_COLUMNS_BASE = [
    "year",
    "student_id",
    "raw_score",
    "cumulative_gpa",
    "current_semester_gpa",
    "previous_courses_taken",
    "previous_credits_earned",
    "expected_difficulty",
    "subject_type",
    "fail_rate_general",
    "fail_rate_major",
    "subject_type_num"
]

# Chỉ drop những cột thực sự tồn tại
DROP_COLUMNS = [c for c in DROP_COLUMNS_BASE if c in df.columns]

# --- 3) Build X, y ---
if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Missing TARGET_COLUMN: {TARGET_COLUMN}")

X = df.drop(columns=DROP_COLUMNS, errors="ignore")
y = df[TARGET_COLUMN].copy()

print("X shape after drop:", X.shape)
print("y shape:", y.shape)

# --- 4) Group split by student_id (if available) ---
if "student_id" in df.columns:
    groups = df["student_id"]
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))
    
    X_train = X.iloc[train_idx].copy()
    X_test  = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_test  = y.iloc[test_idx].copy()
    
    print("Used GroupShuffleSplit by student_id.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print("student_id not found -> used random train_test_split.")

print("Train size:", X_train.shape, "| Test size:", X_test.shape)

# --- 5) Identify feature types ---
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features   = X_train.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical features ({len(categorical_features)}):", categorical_features[:10], "..." if len(categorical_features) > 10 else "")
print(f"Numerical features ({len(numerical_features)}):", numerical_features[:10], "..." if len(numerical_features) > 10 else "")

# --- 6) Build transformers with imputers ---
# OneHotEncoder compatibility across sklearn versions
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe),
])

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# --- 7) ColumnTransformer ---
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numerical_transformer, numerical_features),
    ],
    remainder="drop"
)

# --- 8) Preprocessing pipeline ---
preprocessing_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor)
])

# Fit on train, transform train/test
X_train_transformed = preprocessing_pipeline.fit_transform(X_train)
X_test_transformed  = preprocessing_pipeline.transform(X_test)

# --- 9) Get feature names robustly ---
feature_names = None
try:
    feature_names = preprocessing_pipeline.named_steps["preprocessor"].get_feature_names_out()
except Exception:
    # Fallback manual build
    cat_names = []
    if len(categorical_features) > 0:
        try:
            cat_names = preprocessing_pipeline.named_steps["preprocessor"] \
                .named_transformers_["cat"] \
                .named_steps["onehot"] \
                .get_feature_names_out(categorical_features).tolist()
        except Exception:
            cat_names = [f"cat_{i}" for i in range(
                preprocessing_pipeline.named_steps["preprocessor"]
                .named_transformers_["cat"]
                .named_steps["onehot"]
                .transform(X_train[categorical_features]).shape[1]
            )]
    num_names = numerical_features
    feature_names = np.array(cat_names + num_names, dtype=object)

print("X_train_transformed:", X_train_transformed.shape)
print("X_test_transformed :", X_test_transformed.shape)
print("Total feature names:", len(feature_names) if feature_names is not None else "N/A")

# --- 10) Quick sanity checks ---
print("\nSample of feature names:")
if feature_names is not None:
    print(feature_names[:30])

# Export variables for later cells
# X_train_transformed, X_test_transformed, y_train, y_test, feature_names, preprocessing_pipeline


Loaded: clean_student_data_v2.csv
Shape: (340000, 30)
X shape after drop: (340000, 18)
y shape: (340000,)
Used GroupShuffleSplit by student_id.
Train size: (272000, 18) | Test size: (68000, 18)
Categorical features (2): ['course_code', 'study_format'] 
Numerical features (16): ['semester_number', 'credits_unit', 'last_score', 'mean_prev_score', 'score_trend', 'score_stability', 'score_stability_cv', 'recent_improvement', 'n_assessments', 'weekly_study_hours'] ...
X_train_transformed: (272000, 79)
X_test_transformed : (68000, 79)
Total feature names: 79

Sample of feature names:
['cat__course_code_CHE 101' 'cat__course_code_CMU-CS 246'
 'cat__course_code_CMU-CS 252' 'cat__course_code_CMU-CS 297'
 'cat__course_code_CMU-CS 303' 'cat__course_code_CMU-CS 311'
 'cat__course_code_CMU-CS 316' 'cat__course_code_CMU-CS 445'
 'cat__course_code_CMU-CS 447' 'cat__course_code_CMU-CS 462'
 'cat__course_code_CMU-ENG 130' 'cat__course_code_CMU-ENG 230'
 'cat__course_code_CMU-IS 401' 'cat__course_code_C

In [4]:
# Huấn luyện pipeline trên dữ liệu training
print("Đang huấn luyện pipeline trên X_train...")
preprocessing_pipeline.fit(X_train)
print("Huấn luyện hoàn tất.")

# Lưu pipeline đã được huấn luyện ra file joblib
pipeline_filename = 'preprocessing_pipeline_6_12_2025_New.joblib'
joblib.dump(preprocessing_pipeline, pipeline_filename)

print(f"\nToàn bộ pipeline đã được lưu thành công vào file: '{pipeline_filename}'")
display(preprocessing_pipeline)

Đang huấn luyện pipeline trên X_train...
Huấn luyện hoàn tất.

Toàn bộ pipeline đã được lưu thành công vào file: 'preprocessing_pipeline_6_12_2025_New.joblib'


,steps,"[('preprocessor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
